# Seminar 5: RAG as a wildfire research assistant

The aim of our seminar today is to build a **small but complete Retrieval-Augmented Generation (RAG) system**.  
The example is inspired by my environmental and wildfire susceptibility research.

## Practical focus:

1. Build a document QA system from a small corpus.
2. Compare **chunking strategies**.
3. Compare **BM25**, **vector search**, and **hybrid retrieval**.
4. Generate grounded answers with source references.
5. Evaluate retrieval quality with simple metrics.
6. Expose the RAG pipeline with **Gradio**.

## 0. Installing dependencies

Run this cell in Colab or your local environment.
The core notebook uses a small open-source embedding model. Generation can run in two modes:

- **default:** extractive fallback answer, no API key needed
- **optional:** Groq-hosted LLM generation if a Groq API key is available

In [ ]:
# Installing the Python packages used in the notebook:

    # sentence-transformers = text embeddings
    # faiss-cpu = vector similarity search
    # rank_bm25 = keyword-based retrieval
    # groq = optional LLM generation thru the Groq API
    # gradio = simple web interface
    # pandas = tabular data handling
    # numpy = numerical arrays and vector operations

%pip install -q sentence-transformers faiss-cpu rank_bm25 groq gradio pandas numpy

## 1. Imports and setup

In [ ]:
import os                          # working with environment variables
import getpass                     # securely entering passwords/API keys
import re                          # regular expressions for text cleaning
from dataclasses import dataclass  # convenient way to define structured objects
from typing import List, Dict, Tuple  # type hints for clearer inputs/outputs

import numpy as np                 # numerical arrays and calculations
import pandas as pd                # tabular data handling
from rank_bm25 import BM25Okapi    # BM25 keyword-based retrieval
from sentence_transformers import SentenceTransformer  # creating text embeddings
import faiss                       # fast nearest-neighbor search over embeddings

pd.set_option('display.max_colwidth', 120)  # show longer text in pandas tables

In [ ]:
# Optional Groq setup. Press Enter to skip and use extractive generation only

DEFAULT_GROQ_MODEL = "openai/gpt-oss-20b"

if not os.getenv("GROQ_API_KEY"):
    groq_key = ""
    if groq_key:
        os.environ["GROQ_API_KEY"] = groq_key
        print("Groq API key stored for this notebook session.")
    else:
        print("No Groq API key entered; extractive generation remains available.")
else:
    print("GROQ_API_KEY is already available in the environment.")


## 2. A compact research-like corpus

In a real RAG application, these documents might come from PDFs, HTML pages, reports, lab notes, spreadsheets, or internal project documents.
For this seminar, we use short, inspectable documents so that you can see exactly what the retriever is doing.

In [ ]:
# The corpus acts as an external knowledge base for the RAG system

CORPUS = [
    {
        "doc_id": "carpathian_summary",
        "title": "Carpathian fire susceptibility study: overview",
        "source": "Manczinger et al. 2025, Scientific Reports, paraphrased",
        "text": """
        The Carpathian fire susceptibility study developed an interpretable machine learning framework for vegetation fire susceptibility across seven countries of the Carpathian region: Czechia, Hungary, Poland, Romania, Serbia, Slovakia, and Ukraine. The study combined harmonised environmental and socio-economic predictors with feature selection and SHAP-based interpretation. The practical aim was not only to predict fire occurrence, but also to create interpretable susceptibility maps that can support local communities and policy-makers.
        """
    },
    {
        "doc_id": "carpathian_data",
        "title": "Data and predictors",
        "source": "Manczinger et al. 2025, Scientific Reports, paraphrased",
        "text": """
        The framework used twenty-seven predictors covering climatic, vegetation-related, topographic, and socio-economic factors. Examples include temperature, precipitation, NDVI from satellite products, elevation, slope/aspect-related variables, distance to water bodies, and indicators of land cover or human influence. Fire and non-fire observations were harmonised across the region before modelling.
        """
    },
    {
        "doc_id": "carpathian_modelling",
        "title": "Modelling workflow and interpretability",
        "source": "Manczinger et al. 2025, Scientific Reports, paraphrased",
        "text": """
        The modelling workflow used a two-stage strategy. First, feature selection methods such as variance inflation factor filtering, Lasso, and recursive feature elimination were used to refine candidate predictors. Second, machine learning models were trained and evaluated, and SHAP values were used to interpret how predictors contributed to model predictions. This combination of prediction and interpretation is important because environmental risk maps should be explainable to decision-makers.
        """
    },
    {
        "doc_id": "carpathian_drivers",
        "title": "Main environmental drivers",
        "source": "Manczinger et al. 2025, Scientific Reports, paraphrased",
        "text": """
        Higher mean temperature and lower precipitation can increase fire susceptibility because they contribute to drier vegetation and fuels. NDVI variables were negatively associated with fire occurrence: healthier and more vigorous vegetation was generally linked to lower fire susceptibility, whereas low NDVI may indicate stressed, sparse, or previously disturbed vegetation. Greater distance from water bodies was also associated with higher susceptibility, possibly reflecting lower moisture availability.
        """
    },
    {
        "doc_id": "carpathian_maps",
        "title": "Susceptibility maps and H3 grid",
        "source": "Manczinger et al. 2025, Scientific Reports, paraphrased",
        "text": """
        The trained model was used to project fire susceptibility across the Carpathian region using a grid of H3 hexagons. This type of spatial indexing makes it easier to represent predictions consistently across a large region and to identify hotspots. The maps showed that susceptibility is not evenly distributed: lower elevations and foothill areas often showed higher predicted fire probability, while some mountainous areas also contained localized hotspots.
        """
    },
    {
        "doc_id": "carpathian_policy",
        "title": "Management interpretation",
        "source": "Manczinger et al. 2025, Scientific Reports, paraphrased",
        "text": """
        The study highlights that fire susceptibility is a cross-border environmental risk. Because the Carpathians span several countries, mitigation may require coordinated early-warning systems, shared monitoring, fuel management, and rapid response planning. Susceptibility maps are not deterministic forecasts; they should be treated as decision-support layers that indicate where prevention and monitoring may be prioritized.
        """
    },
    {
        "doc_id": "limitations",
        "title": "Important limitations of susceptibility modelling",
        "source": "Additional note",
        "text": """
        Fire susceptibility modelling estimates where conditions resemble locations with historical fires. It is different from short-term fire forecasting, which would require near real-time weather, ignition sources, human activity, and fuel moisture. A high AUC can indicate strong discrimination in the dataset, but operational usefulness also depends on calibration, spatial transferability, temporal validation, uncertainty, and whether decision-makers can act on the predictions.
        """
    },
    {
        "doc_id": "rag_definition",
        "title": "RAG definition",
        "source": "AI Engineering Ch. 6, paraphrased note",
        "text": """
        Retrieval-Augmented Generation is a pattern in which a system first retrieves relevant information from external memory and then gives this information to a generative model. The goal is to help the model answer using context that may be too large, too private, or too recent to be stored in the model's parameters. The retriever is therefore a critical component of the whole system.
        """
    },
    {
        "doc_id": "rag_chunking",
        "title": "Chunking trade-offs",
        "source": "AI Engineering Ch. 6, paraphrased note",
        "text": """
        Chunking determines what units are indexed and retrieved. Smaller chunks can be more precise and allow more diverse pieces of evidence to fit into the model context, but they may lose surrounding information. Larger chunks preserve more context but may retrieve irrelevant text and increase the token budget. There is no universally best chunk size; it should be tested for the document type and user questions.
        """
    },
    {
        "doc_id": "rag_retrieval",
        "title": "Retrieval methods",
        "source": "AI Engineering Ch. 6, paraphrased note",
        "text": """
        Term-based retrieval such as BM25 is fast, simple, and strong for exact keywords. Embedding-based retrieval uses semantic similarity and can find relevant text even when the query uses different wording. Hybrid search combines multiple retrievers and can be followed by reranking. Retrieval should be evaluated directly, not only through the final generated answer.
        """
    },
    {
        "doc_id": "rag_evaluation",
        "title": "RAG evaluation",
        "source": "AI Engineering Ch. 6 and note, paraphrased",
        "text": """
        RAG evaluation should consider both retrieval quality and generation quality. Retrieval can be measured using context precision, context recall, mean reciprocal rank, or related ranking metrics. Generation should be checked for faithfulness, correctness, relevance, and whether claims are supported by the retrieved context. A good RAG answer should make it possible to inspect the sources.
        """
    },
    {
        "doc_id": "rag_security",
        "title": "RAG risks and security",
        "source": "Additional note",
        "text": """
        RAG systems introduce new risks because retrieved documents become part of the model input. A malicious document can contain prompt injection instructions. Private documents can leak if access control is not enforced. Systems should treat retrieved text as untrusted data, cite sources, separate instructions from evidence, and refuse to answer when the retrieved context is insufficient.
        """
    }
]

corpus_df = pd.DataFrame(CORPUS)
corpus_df[["doc_id", "title", "source"]]

## 3. Chunking strategies

We compare two simple strategies:

- **paragraph chunks:** we keep each document as a meaningful paragraph-level chunk;
- **fixed-size word chunks:** split into approximately equal word windows with overlap.

For real PDFs or reports, you would usually add metadata such as section title, page number, author, date, and document type.

In [ ]:
# 1. A helper function is defined to normalize whitespace before text is chunked or indexed:

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

# 2. We also define the structure used to store one retreivable text chunk and its metadata:

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    source: str
    text: str
    strategy: str

# 3. We here define a chunking strategy that keeps each short source document as one chunk (=paragraph chunk):

def paragraph_chunks(corpus: List[Dict]) -> List[Chunk]:
    chunks = []
    for doc in corpus:
        txt = clean_text(doc["text"])
        chunks.append(Chunk( # creating a Chunk object with cleaned text and metadata
            chunk_id=f"{doc['doc_id']}::p0",
            doc_id=doc["doc_id"],
            title=doc["title"],
            source=doc["source"],
            text=txt,
            strategy="paragraph"
        ))
    return chunks

# 4. And here we define a sliding window chunker based on word counts and overlap (=fixed word chunk):

def fixed_word_chunks(corpus: List[Dict], chunk_words: int = 70, overlap_words: int = 15) -> List[Chunk]:
    chunks = []
    for doc in corpus:
        words = clean_text(doc["text"]).split()
        start = 0 # start position of the first chunk
        idx = 0   # starts the chunk counter at 0
        step = max(1, chunk_words - overlap_words) # how far the chunking window moves each time

        # Keep making chunks until the whole doc processed
        while start < len(words):
            end = min(len(words), start + chunk_words)
            chunk_text = " ".join(words[start:end])
            chunks.append(Chunk(
                chunk_id=f"{doc['doc_id']}::w{idx}",
                doc_id=doc["doc_id"],
                title=doc["title"],
                source=doc["source"],
                text=chunk_text,
                strategy=f"fixed_{chunk_words}w_overlap_{overlap_words}w"
            ))
            if end == len(words): # if we reach the end of doc, stop chunking
                break
            start += step         # if not, move the window forward by 55 words 
            idx += 1
    return chunks

paragraph = paragraph_chunks(CORPUS)
fixed = fixed_word_chunks(CORPUS, chunk_words=70, overlap_words=15)

print(f"Paragraph chunks: {len(paragraph)}")
print(f"Fixed-size chunks: {len(fixed)}")
pd.DataFrame([c.__dict__ for c in fixed[:6]])[["chunk_id", "doc_id", "strategy", "text"]]

## 4. Building retrievers

We will implement three retrieval modes:

1. **BM25:** lexical keyword retrieval.
2. **Vector search:** semantic retrieval using sentence-transformer embeddings and FAISS.
3. **Hybrid RRF:** combines BM25 and vector rankings using reciprocal rank fusion.

In [ ]:
# 1. We define a simple lowercase alhanumeric tokenizer for BM25 (still not a sophisticated NLP tokenizer but for our case is acceptable now!)

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())

# 2. Here we define a reusable retireval index called RAGIndex which builds two different search engines over the same chunks:

class RAGIndex:
    def __init__(
            self, # a particular RAGIndex object, e.g. paragraph
            chunks: List[Chunk],
            model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
            ):
        
        self.chunks = chunks
        self.texts = [c.text for c in chunks]
        self.chunk_lookup = {c.chunk_id: c for c in chunks} # creating a dict

        # 1 - Building the BM25 index
        self.tokens = [tokenize(t) for t in self.texts]
        self.bm25 = BM25Okapi(self.tokens)

        # 2 - Building the embedding/vector index
        self.embedder = SentenceTransformer(model_name)
        self.embeddings = self.embedder.encode(self.texts, normalize_embeddings=True, show_progress_bar=False)
        self.embeddings = np.asarray(self.embeddings, dtype="float32") # float32 format is needed for FAISS
            # Creating the FAISS index
            # Inner product equals cosine similarity because embeddings are normalized
        self.faiss_index = faiss.IndexFlatIP(self.embeddings.shape[1])
        self.faiss_index.add(self.embeddings)

    # Lexical retrieval using BM25 scores

    def bm25_search(self, query: str, k: int = 5) -> List[Tuple[Chunk, float]]:
        k = min(int(k), len(self.chunks)) # prevents requesting more chunks than what we have
        scores = self.bm25.get_scores(tokenize(query)) # you get a score for every chunk
        idxs = np.argsort(scores)[::-1][:k] # sorts the chunks from highest score to lowest score and keep the best k ([::-1] reverses the sorting)
        return [(self.chunks[i], float(scores[i])) for i in idxs]

    # Vector search
    
    def vector_search(self, query: str, k: int = 5) -> List[Tuple[Chunk, float]]:
        k = min(int(k), len(self.chunks))
        q = self.embedder.encode([query], normalize_embeddings=True, show_progress_bar=False).astype("float32")
        scores, idxs = self.faiss_index.search(q, k) # FAISS finds the k document vectors closest to the query vector
        return [(self.chunks[int(i)], float(s)) for i, s in zip(idxs[0], scores[0])]

    # Hybrid search
    
    def hybrid_rrf(
            self,
            query: str,
            k: int = 5,
            fetch_k: int = 8, # retrieve 8 candidates from each search method, combine them, then return the final 5 best
            rrf_k: int = 6
            ) -> List[Tuple[Chunk, float]]:
        k = min(int(k), len(self.chunks))
        fetch_k = min(max(int(fetch_k), k), len(self.chunks))

        bm25_ranked = self.bm25_search(query, k=fetch_k)
        vector_ranked = self.vector_search(query, k=fetch_k)
        
        scores = {}
        chunk_by_id = {}
        
        for ranked in [bm25_ranked, vector_ranked]:
            for rank, (chunk, _) in enumerate(ranked, start=1):
                scores[chunk.chunk_id] = scores.get(chunk.chunk_id, 0.0) + 1.0 / (rrf_k + rank)
                chunk_by_id[chunk.chunk_id] = chunk
        
        ranked_ids = sorted(scores, key=scores.get, reverse=True)[:k]
        return [(chunk_by_id[cid], scores[cid]) for cid in ranked_ids]

# 3. Now having a fully searchable RAGindex for the paragraph & fized-size chunking strategies
    # What does this mean?
        # 1 - We store the paragraph chunks,
        # 2 - We tokenize them for BM25,
        # 3 - We build the BM25 index,
        # 4 - We create embeddings,
        # 5 - We convert them to float32,
        # 6 - We build the FAISS vector index,
        # 7 - We store the embeddings in FAISS.

print("Building paragraph index...")
paragraph_index = RAGIndex(paragraph)

print("Building fixed-size index...")
fixed_index = RAGIndex(fixed)

print("Done.")

## 5. Inspecting retrieval results

Good RAG development always starts by looking at **what the retriever actually returns**.

In [ ]:
# Creating a table for retrieval results

def show_results(results: List[Tuple[Chunk, float]], max_chars: int = 260):
    rows = []
    for rank, (chunk, score) in enumerate(results, start=1):
        rows.append({
            "rank": rank,
            "score": round(score, 4),
            "doc_id": chunk.doc_id,
            "title": chunk.title,
            "strategy": chunk.strategy,
            "text": chunk.text[:max_chars] + ("..." if len(chunk.text) > max_chars else "")
        })
    return pd.DataFrame(rows)

# Now we are sending the same question to all 3 retrieval methods

query = "Which environmental variables increase fire susceptibility in the Carpathians?"

print("BM25 results")
display(show_results(paragraph_index.bm25_search(query, k=5)))

print("Vector results")
display(show_results(paragraph_index.vector_search(query, k=5)))

print("Hybrid RRF results")
display(show_results(paragraph_index.hybrid_rrf(query, k=5)))

# Note: do not compare the resulting scores between the retrieval methods as they are completely different scoring systems!!!

### Exercise 1

Try the following queries and compare BM25 vs vector search:

- `Why can NDVI be negatively associated with fire occurrence?`
- `How is susceptibility different from real-time fire forecasting?`
- `How should RAG be evaluated?`
- `What risks are introduced when retrieved documents are added to the prompt?`

Which retrieval method works best for exact terms? Which works best for paraphrased questions?

## 6. Simple grounded answer generation

A production RAG system usually calls an LLM with a prompt containing the question and retrieved context. To keep this notebook runnable without an API key, I also provide a deterministic **extractive fallback**. If a Groq API key is available, the same retrieved context can be passed to a Groq-hosted LLM.

In [ ]:
# 1 - Preparing retrieved chunks for the LLM

def format_context(results: List[Tuple[Chunk, float]], max_chars_per_chunk: int = 700) -> str:
    parts = []
    for i, (chunk, score) in enumerate(results, start=1):
        txt = chunk.text[:max_chars_per_chunk]
        parts.append(
            f"[S{i}] {chunk.title} | {chunk.source} | doc_id={chunk.doc_id}\n{txt}"
        )
    return "\n\n".join(parts)

# Optional: answering without an LLM API key
def extractive_answer(
    question: str,
    results: List[Tuple[Chunk, float]],
    max_sentences: int = 5
) -> str:
    query_terms = set(tokenize(question))
    candidates = []

    for source_idx, (chunk, score) in enumerate(results, start=1):
        sentences = re.split(r"(?<=[.!?])\s+", chunk.text)
        
        # Comparing each sentence with the question
        for sent in sentences:
            sent_terms = set(tokenize(sent))
            overlap = len(query_terms & sent_terms) # & means intersection
            candidates.append(
                (overlap, score, source_idx, chunk.doc_id, sent.strip())
            )
    
    # Ranking the candidate sentences
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    selected = [c for c in candidates if c[0] > 0][:max_sentences]

    if not selected:
        return "I did not find enough supporting context in the retrieved documents."

    bullets = [
        f"- {sent} [S{source_idx}]"
        for _, _, source_idx, _, sent in selected
    ]
    sources = sorted({
        f"[S{source_idx}] {doc_id}"
        for _, _, source_idx, doc_id, _ in selected
    })

    return (
        "Answer based on retrieved context:\n"
        + "\n".join(bullets)
        + "\n\nSources used: "
        + "; ".join(sources)
    )

# 2 - Actual LLM generation step

def llm_answer_with_groq(
    question: str,
    results: List[Tuple[Chunk, float]],
    model: str = DEFAULT_GROQ_MODEL
) -> str:

    # If no Groq API key is available
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        return extractive_answer(question, results)

    from groq import Groq
    client = Groq(api_key=api_key)

    context = format_context(results)

    # Critical augmentation step of RAG:
    prompt = f"""
You are a careful scientific teaching assistant.
Answer the question using only the retrieved context below.
If the context is insufficient, say so.
Cite sources using [S1], [S2], etc.

Question:
{question}

Retrieved context:
{context}
""".strip()

    # Sending into the LLM
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0 # asks for deterministic output!
    )

    return response.choices[0].message.content


# Joining retrieval and generation
def rag_answer(
    question: str,
    index: RAGIndex,
    mode: str = "hybrid",
    k: int = 4,
    generator: str = "extractive",
    groq_model: str = DEFAULT_GROQ_MODEL
):

    # 1. RETRIEVAL
    if mode == "bm25":
        results = index.bm25_search(question, k=k)

    elif mode == "vector":
        results = index.vector_search(question, k=k)

    elif mode == "hybrid":
        results = index.hybrid_rrf(question, k=k)

    else:
        raise ValueError("mode must be one of: bm25, vector, hybrid")

    # 2. GENERATION
    if generator == "extractive":
        answer = extractive_answer(question, results)

    elif generator == "groq":
        answer = llm_answer_with_groq(
            question,
            results,
            model=groq_model
        )

    else:
        raise ValueError("generator must be 'extractive' or 'groq'")

    return answer, results

# Example question
question = "Why might NDVI be negatively associated with fire occurrence?"

# Use Groq if a key was supplied
generator_mode = "groq" if os.getenv("GROQ_API_KEY") else "extractive"

answer, results = rag_answer(
    question,
    paragraph_index,
    mode="hybrid",
    k=4,
    generator=generator_mode
)

print(f"Generator: {generator_mode}")
print(answer)
display(show_results(results))

## 7. Comparing chunking strategies

The same query can retrieve different evidence depending on chunking. In this small corpus, the differences are subtle, but in long PDFs or reports they can be decisive.

In [ ]:
# Does the chunking strategy change what our RAG system retrieves and answers?

def compare_chunking(
        question: str,
        mode: str = "hybrid",
        k: int = 4
        ):
    answer_p, results_p = rag_answer(question, paragraph_index, mode=mode, k=k, generator="groq")
    answer_f, results_f = rag_answer(question, fixed_index, mode=mode, k=k, generator="groq")

    print("QUESTION:", question)
    print("\n--- Paragraph chunks ---")
    display(show_results(results_p))
    print(answer_p)
    print("\n--- Fixed-size chunks ---")
    display(show_results(results_f))
    print(answer_f)

compare_chunking("What is the difference between fire susceptibility mapping and operational fire forecasting?", mode="hybrid", k=4)

### Exercise 2

Change `chunk_words` and `overlap_words` in the fixed-size chunker. Rebuild the index and test again.

Questions to discuss:

- Does the answer become more specific or more noisy?
- Which chunks would fit better into an LLM prompt?
- When would paragraph-based chunking be better than fixed-size chunking?

## 8. Evaluating retrieval with small gold labels

We create a tiny evaluation set. In a real project, this evaluation set should be curated carefully and expanded over time (as in your project work).

In [ ]:
# Small evaluation set

EVAL_SET = {
    "Which variables are associated with higher fire susceptibility?": {"carpathian_drivers", "carpathian_data"},
    "How were susceptibility maps represented spatially?": {"carpathian_maps"},
    "Why is susceptibility not the same as short-term forecasting?": {"limitations", "carpathian_policy"},
    "What are the differences between BM25 and vector retrieval?": {"rag_retrieval"},
    "How should a RAG system be evaluated?": {"rag_evaluation"},
    "What security risks can RAG introduce?": {"rag_security"},
}

# Helper running retrievals and returns only their document IDs for evaluation
def retrieve_doc_ids(
        index: RAGIndex,
        question: str,
        mode: str,
        k: int = 4
        ) -> List[str]:
    if mode == "bm25":
        results = index.bm25_search(question, k=k)
    elif mode == "vector":
        results = index.vector_search(question, k=k)
    elif mode == "hybrid":
        results = index.hybrid_rrf(question, k=k)
    else:
        raise ValueError(mode)
    return [chunk.doc_id for chunk, _ in results]

# Calculating precision@k, recall@k, and reciprocal rank

def precision_recall_rr(
        retrieved_doc_ids: List[str],
        relevant_doc_ids: set,
        k: int
        ):
    
    retrieved_k = retrieved_doc_ids[:k]
    
    # Converting retrieved list into correct/incorrect indicators
    hits = [
        1 if doc_id in relevant_doc_ids else 0
        for doc_id in retrieved_k
        ]

    # Precision = Of the documents I retrieved, how many were relevant?
    precision_at_k = sum(hits) / k

    # Recall = Of all the relevant documents that exist, how many did we successfully retrieve?
    recall_at_k = len(set(retrieved_k) & relevant_doc_ids) / len(relevant_doc_ids)

    # Reciprocal Rank = How early does the first relevant result appear? (= the metric strongly rewards putting a correct document near the very top)
    rr = 0.0
    
    for idx, hit in enumerate(hits, start=1):
        if hit:
            rr = 1.0 / idx
            break
    return precision_at_k, recall_at_k, rr

# Now we evaluate one retrieval method across every question

def evaluate(
        index: RAGIndex,
        mode: str,
        k: int = 4
        ) -> pd.DataFrame:
    rows = []

    for question, relevant in EVAL_SET.items():
        retrieved = retrieve_doc_ids(index, question, mode, k=k)
        p, r, rr = precision_recall_rr(retrieved, relevant, k)
        rows.append({
            "question": question,
            "mode": mode,
            "precision@k": round(p, 3),
            "recall@k": round(r, 3),
            "RR": round(rr, 3),
            "relevant": ", ".join(sorted(relevant)),
            "retrieved": ", ".join(retrieved)
        })

    return pd.DataFrame(rows)

# Comparing all 3 retrieval approaches

for mode in ["bm25", "vector", "hybrid"]:
    print("\n", mode.upper())
    display(evaluate(paragraph_index, mode=mode, k=4))

## 9. A simple Gradio app

Now we transfer the RAG result to a small web interface. In Colab, set `share=True` if you want a temporary public link.

In [ ]:
# Importing Gradio, which wraps the RAG pipeline in a simple interactive web interface
import gradio as gr

# Mapping interface labels to the corresponding pre-built RAG indexes
INDEXES = {
    "paragraph": paragraph_index,
    "fixed-size": fixed_index,
}

# Function called whenever the user submits a question
def gradio_rag(
    question,
    chunking_strategy,
    retrieval_mode,
    top_k,
    use_groq
):
    index = INDEXES[chunking_strategy]

    generator_mode = "groq" if use_groq else "extractive"

    answer, results = rag_answer(
        question,
        index,
        mode=retrieval_mode,
        k=int(top_k),
        generator=generator_mode,
        groq_model=DEFAULT_GROQ_MODEL
    )

    table = show_results(results, max_chars=320)
    return answer, table

# Building the actual webpage

with gr.Blocks(title="Wildfire RAG Research Assistant") as demo:
    gr.Markdown("# Wildfire RAG Research Assistant")
    gr.Markdown(
        "Ask questions over a small research-like corpus. "
        "Inspect the retrieved chunks before trusting the answer."
    )

    with gr.Row():
        question = gr.Textbox(
            label="Question",
            value=(
                "Which variables are associated with higher "
                "fire susceptibility in the Carpathians?"
            ),
            lines=2
        )

    with gr.Row():
        chunking = gr.Radio(
            ["paragraph", "fixed-size"],
            label="Chunking strategy",
            value="paragraph"
        )

        mode = gr.Radio(
            ["bm25", "vector", "hybrid"],
            label="Retrieval mode",
            value="hybrid"
        )

        top_k = gr.Slider(
            1,
            8,
            value=4,
            step=1,
            label="Top-k chunks"
        )

        use_groq = gr.Checkbox(
            label="Use Groq LLM (requires a Groq API key)",
            value=False
        )

    run = gr.Button("Run RAG")
    answer_box = gr.Textbox(label="Answer", lines=10)
    retrieved_table = gr.Dataframe(label="Retrieved chunks")

    run.click(
        gradio_rag,
        inputs=[question, chunking, mode, top_k, use_groq],
        outputs=[answer_box, retrieved_table]
    )


# In Colab, change share=False to share=True if you need a temporary public link.
demo.launch(share=False)

## 10. Your task

Choose one:

1. **Add documents** from your own project topic and test whether retrieval still works.
2. **Change chunking** and explain how it affects one query.
3. **Create a failure case** where the system retrieves the wrong context or has no evidence.
4. **Add metadata filtering** such as source type, country, method, or year.
5. **Add a reranker** or compare a larger embedding model.

Suggested short report format:

- Query:
- Retrieval mode and chunking:
- Retrieved chunks:
- Answer:
- What worked:
- What failed:
- Improvement idea: